In [1]:
import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageFilter, ImageEnhance
import os
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler
import random
from concurrent.futures import ThreadPoolExecutor

## chuyển đổi hình ảnh trên tập dữ liệu CIC DDOS 2019

In [2]:

# selected_columns = ['Flow Duration','Fwd Packets Length Total',
#                     'Fwd Packet Length Max',
#                     'Fwd Packet Length Min', 'Fwd Packet Length Mean',
#                     'Flow Bytes/s', 'Flow Packets/s',
#                     'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 
#                     'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
#                     'Fwd IAT Min',  'Bwd IAT Mean', 'Bwd IAT Std',
#                     'Bwd IAT Max', 'Bwd IAT Min', 'Bwd PSH Flags',
#                     'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
#                     'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
#                     'Packet Length Std', 'Packet Length Variance', 
#                     'Avg Packet Size', 'Avg Fwd Segment Size',
#                     'Subflow Fwd Packets', 'Subflow Fwd Bytes',
#                     'Fwd Seg Size Min',
#                     'Idle Mean', 'Idle Std', 'Idle Max', 'Label']

# selected_columns = ['Total Fwd Packets', 'Total Backward Packets',
#        'Fwd Packets Length Total', 'Bwd Packets Length Total',
#        'Fwd Packet Length Max', 'Fwd Packet Length Min',
#        'Fwd Packet Length Mean', 'Fwd Packet Length Std',
#        'Bwd Packet Length Max', 'Bwd Packet Length Min',
#        'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
#        'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
#        'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
#        'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
#        'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
#        'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
#        'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
#        'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
#        'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
#        'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count',
#        'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio',
#        'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size',
#        'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate',
#        'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate',
#        'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets',
#        'Subflow Bwd Bytes', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes',
#        'Fwd Act Data Packets', 'Fwd Seg Size Min', 'Active Mean', 'Active Std',
#        'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max',
#        'Idle Min', 'Label']

# selected_columns = ['ACK Flag Count', 'URG Flag Count', 'Fwd Packets Length Total', 'Init Fwd Win Bytes', 
#                     'CWE Flag Count', 'Fwd PSH Flags', 'Flow Duration', 
#                     'Packet Length Std', 'Fwd IAT Total', 'Flow IAT Std', 'Fwd Header Length', 'Fwd Act Data Packets', 
#                     'Bwd Packet Length Min', 'Bwd Packet Length Std', 'Fwd Seg Size Min', 'Init Bwd Win Bytes', 
#                     'Fwd Packet Length Std', 'Total Fwd Packets', 'Total Backward Packets', 'Bwd Packets Length Total','Label']



selected_columns = ['Total Fwd Packets', 'Total Backward Packets',
       'Fwd Packets Length Total', 'Bwd Packets Length Total',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 
       'CWE Flag Count', 'ECE Flag Count',
       'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size',
       'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 
       'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 
       'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets',
       'Subflow Bwd Bytes', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes',
       'Fwd Act Data Packets', 'Fwd Seg Size Min', 'Active Mean', 'Active Std',
       'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max',
       'Idle Min', 'Label']

len(selected_columns)

65

In [3]:
df = pd.read_csv('data/csv/cicddos_2019.csv')
df = df[selected_columns]

In [6]:
# datadir = 'cic_ddos_2019_images'
# def convert(df_normalized_splited, label, num):
#     # Chuyển mỗi dòng thành ma trận ảnh (giả sử 16x16 pixels)
#     image_size = (5, 15)

#     # lưu ảnh tập train
#     i = 1
#     for row in df_normalized_splited.values:

#         image_array = np.array(row).reshape(image_size)  # hiện tại chọn 75 đặc trưng nên chọn reshape với kích thước 5x15 = 75
#         image_array = np.nan_to_num(image_array)  # Thay thế các giá trị không hợp lệ bằng 0
#         image = Image.fromarray((image_array * 255).astype(np.uint8)) # nhân 255 để chuyển sang giá trị RGB
#         image = image.convert("RGB")
        
#         # image = image.resize((224, 224))  # Chuyển đổi kích thước hình ảnh thành 224x224 (thường train model dùng kích thước này)
#         image.save(f"data/{datadir}/{label}/{str(i)}.png")

#         i = i + 1
#         if i > num:
#             break


datadir = 'cic_ddos_2019_images'
def convert(df_normalized_splited, label, num):
    # Kích thước ảnh ban đầu (5x15) => Mở rộng mỗi điểm thành 3x3 pixel => Ảnh mới (15x45)
    image_size = (8, 8) # kích thước ảnh ban đầu
    upscale_factor = 28 # tỉ lệ tăng kích thước điểm ảnh => size ảnh: (8x28) x (8x28)
    new_image_size = (image_size[0] * upscale_factor, image_size[1] * upscale_factor)

    os.makedirs(f"data/{datadir}/{label}", exist_ok=True)  # Tạo thư mục nếu chưa có

    i = 1
    for row in df_normalized_splited.values:
        # Chuyển đổi dòng thành ma trận ảnh ban đầu (8x8)
        image_array = np.array(row).reshape(image_size)
        image_array = np.nan_to_num(image_array)  # Thay thế giá trị NaN bằng 0

        # Phóng to mỗi pixel np.kron()
        upscale_matrix = np.ones((upscale_factor, upscale_factor))
        enlarged_image_array = np.kron(image_array, upscale_matrix)

        # Chuyển đổi sang ảnh
        image = Image.fromarray((enlarged_image_array * 255).astype(np.uint8))  # Chuyển sang RGB
        image = image.convert("RGB")

        
        ###### thêm nhiễu vào ảnh ######

        # Xoay ảnh ngẫu nhiên (-15° đến 15°)
        rotate_prob = random.random() < 0.6
        if rotate_prob:  
            angle = random.uniform(-50, 50)  
            image = image.rotate(angle)


        # Lật ảnh ngẫu nhiên
        flip_prob = random.random() < 0.6
        if flip_prob:
            if random.random() < 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)  # Lật ngang
            else:
                image = image.transpose(Image.FLIP_TOP_BOTTOM)  # Lật dọc

        # Điều chỉnh độ sáng ngẫu nhiên (từ 70% đến 130%)
        brightness_prob = random.random() < 0.6
        if brightness_prob:
            enhancer = ImageEnhance.Brightness(image)
            factor = random.uniform(0.7, 1.6)  # Thay đổi độ sáng từ 70% đến 130%
            image = enhancer.enhance(factor)


        # Xác suất ngẫu nhiên để thêm hiệu ứng (30% ảnh bị làm mờ, 30% ảnh bị nhiễu)
        blur_prob = random.random() <= 0.6  # 30% xác suất làm mờ
        noise_prob = random.random() <= 0.6  # 30% xác suất thêm nhiễu

        '''
        chỉ làm mờ: 0.3, không mờ 0,7 ==> 0.21 xác xuất chỉ mờ
        chỉ làm nhiễu: 0.3, không nhiễu 0,7 ==> 0.21 xác xuất chỉ nhiễu
        vừa mờ: 0.3, vừa nhiễu 0.3 ==> xác xuất vừa mờ vừa nhiễu: 0.09
        không mờ: 0.7, không nhiễu: 0.7 ==> xác xuất không mờ không nhiễu: 0.49
        '''

        # làm mờ ảnh
        if blur_prob:
            image = image.filter(ImageFilter.GaussianBlur(radius=random.uniform(10, 20))) # mức độ mờ từ 1 đến 5

        #làm nhiễu ảnh
        if noise_prob:
            noise = np.random.normal(0, 50, (new_image_size[0], new_image_size[1]))  # Thêm nhiễu Gaussian
            noisy_image_array = np.array(image.convert("L")) + noise  # Chuyển sang grayscale trước khi thêm nhiễu
            noisy_image_array = np.clip(noisy_image_array, 0, 255).astype(np.uint8)  # Giữ giá trị trong khoảng 0-255
            image = Image.fromarray(noisy_image_array).convert("RGB")

        # Lưu ảnh
        image.save(f"data/{datadir}/{label}/{str(i)}.png")

        i += 1
        if i > num:
            break

def setup_to_convert(df_normalized, label):
    os.makedirs(f'data/{datadir}/{label}', exist_ok=True)

    # tổng số lượng ảnh train + valid + test của mỗi nhãn
    n = 200
    convert(df_normalized, label, num=n)

In [7]:
# Nhóm dữ liệu theo cột 'Label'
grouped = df.groupby('Label')

# Tạo dictionary để lưu các DataFrame tương ứng với từng nhãn
dfs = {label: group for label, group in grouped}

def process_label(label, df_label):
    df_drop_label = df_label.drop(columns=['Label'])
    
    df_drop_label.replace([np.inf, -np.inf], np.nan, inplace=True)  # Thay giá trị vô hạn bằng NaN
    df_drop_label.fillna(df_drop_label.median(), inplace=True)  # Điền giá trị NaN bằng giá trị trung vị

    data_features = np.log1p(df_drop_label + 1)
    data_standardized = (data_features - data_features.mean()) / data_features.std()
    data_normalized = data_standardized

    setup_to_convert(data_normalized, label)

# Sử dụng ThreadPoolExecutor để chạy đa luồng với tối đa 5 luồng
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(process_label, label, df_label) for label, df_label in dfs.items()]

    # Đợi tất cả các task hoàn thành
    for future in futures:
        future.result()

print("Hoàn thành xử lý đa luồng!")

f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: divide by zero encountered in log1p
  result = func(self.values, **kwargs)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: divide by zero encountered in log1p
  result = func(self.values, **kwargs)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs

Hoàn thành xử lý đa luồng!
